In [ ]:
# ---- Imports e configuração ----
import math

import folium
import requests

URL = "https://api.olhovivo.sptrans.com.br/v2.1"
sessao = requests.Session()  # se o notebook do grupo já tem URL e sessao, apague estas duas linhas


class ErroViagem(Exception):
    """Erro tratado do sistema (a mensagem já vai pronta para o usuário)."""

In [ ]:
# ---- Req. 07: tratamento de erros ----
def requisitar(caminho, metodo="GET", **params):
    """Chama a API tratando falha de rede, status HTTP, JSON inválido e resposta vazia."""
    try:
        resposta = sessao.request(metodo, f"{URL}{caminho}", params=params, timeout=10)
        resposta.raise_for_status()
        return resposta.json()
    except requests.RequestException as e:
        raise ErroViagem(f"Falha na API em {caminho}: {e}")
    except ValueError:
        raise ErroViagem(f"A API retornou uma resposta inválida em {caminho}.")


def validar_coord(coord, nome):
    """Garante que a coordenada é (lat, lon) numérica e dentro do intervalo válido."""
    try:
        lat, lon = float(coord[0]), float(coord[1])
    except (TypeError, ValueError, IndexError):
        raise ErroViagem(f"Coordenada de {nome} inválida: {coord}")
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ErroViagem(f"Coordenada de {nome} fora do intervalo válido: {coord}")
    return lat, lon


def exigir(dados, mensagem):
    """Resposta vazia ou nenhuma linha/parada encontrada -> ErroViagem."""
    if not dados:
        raise ErroViagem(mensagem)
    return dados


def buscar_previsao(cp, cl):
    """Horários de chegada. Devolve [] se não houver previsão (não é erro fatal)."""
    dados = requisitar("/Previsao", codigoParada=cp, codigoLinha=cl)
    try:
        return [v["t"] for linha in dados["p"]["l"] for v in linha["vs"]]
    except (KeyError, TypeError):
        return []


def distancia_km(a, b):
    """Distância em km entre dois pontos (lat, lon) - Haversine. Usada nas distâncias do resumo."""
    la1, lo1, la2, lo2 = map(math.radians, (*a, *b))
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(h))

In [ ]:
# ---- Req. 05: resumo da viagem ----
def exibir_resumo(v):
    l, emb, des = v["linha"], v["embarque"], v["desembarque"]
    sentido = f"{l['tp']} → {l['ts']}" if l["sl"] == 1 else f"{l['ts']} → {l['tp']}"
    previsao = ", ".join(v["previsao"][:3]) if v["previsao"] else "sem previsão no momento"

    print("=" * 50)
    print("RESUMO DA VIAGEM")
    print("=" * 50)
    print(f"Linha:        {l['lt']}-{l['tl']} ({sentido})")
    print(f"Embarque:     {emb['np']} - {emb['ed']}")
    print(f"Desembarque:  {des['np']} - {des['ed']}")
    print(f"Previsão:     {previsao}")
    print("-" * 50)
    print(f"Origem → embarque:        {v['d_emb']:.2f} km a pé")
    print(f"Trecho de ônibus:         {v['d_onibus']:.2f} km")
    print(f"Desembarque → destino:    {v['d_des']:.2f} km a pé")
    print(f"Total de caminhada:       {v['caminhada']:.2f} km")
    print("=" * 50)

In [ ]:
# ---- Req. 06: mapa interativo ----
def gerar_mapa(origem, destino, v, arquivo="mapa_viagem.html"):
    m = folium.Map(location=origem, zoom_start=13)

    # traçado e paradas da linha
    folium.PolyLine([p["pos"] for p in v["paradas"]], color="blue", weight=3, opacity=0.6).add_to(m)
    for p in v["paradas"]:
        folium.CircleMarker(p["pos"], radius=3, color="gray", fill=True, tooltip=p["np"]).add_to(m)

    # origem, embarque, desembarque e destino
    pontos = [
        (origem, "Origem", "green", "home"),
        (v["embarque"]["pos"], f"Embarque: {v['embarque']['np']}", "blue", "arrow-up"),
        (v["desembarque"]["pos"], f"Desembarque: {v['desembarque']['np']}", "orange", "arrow-down"),
        (destino, "Destino", "red", "flag"),
    ]
    for pos, texto, cor, icone in pontos:
        folium.Marker(pos, tooltip=texto, popup=texto,
                      icon=folium.Icon(color=cor, icon=icone, prefix="fa")).add_to(m)

    # caminhadas (tracejado)
    folium.PolyLine([origem, v["embarque"]["pos"]], color="green", dash_array="5").add_to(m)
    folium.PolyLine([v["desembarque"]["pos"], destino], color="red", dash_array="5").add_to(m)

    m.fit_bounds([origem, destino])
    m.save(arquivo)
    return m

In [ ]:
# ---- TESTE: dados de exemplo ----
# Dados de exemplo (sem API) no formato que o código do grupo deve entregar em `viagem`
origem = (-23.5453, -46.6398)
destino = (-23.5222, -46.7009)

paradas = [
    {"np": f"Parada {i + 1}", "ed": f"Rua Exemplo, {100 * (i + 1)}", "cp": 1000 + i, "pos": pos}
    for i, pos in enumerate([
        (-23.5450, -46.6400), (-23.5400, -46.6500), (-23.5350, -46.6650),
        (-23.5300, -46.6800), (-23.5250, -46.6950), (-23.5220, -46.7005),
    ])
]
emb, des = paradas[0], paradas[-1]

viagem = {
    "linha": {"cl": 2506, "lt": "8000", "tl": 10, "sl": 1, "tp": "PÇA. RAMOS DE AZEVEDO", "ts": "TERM. LAPA"},
    "paradas": paradas,
    "embarque": emb,
    "desembarque": des,
    "d_emb": distancia_km(origem, emb["pos"]),
    "d_des": distancia_km(des["pos"], destino),
    "d_onibus": sum(distancia_km(a["pos"], b["pos"]) for a, b in zip(paradas, paradas[1:])),
    "previsao": ["14:05", "14:18", "14:31"],
}
viagem["caminhada"] = viagem["d_emb"] + viagem["d_des"]

In [ ]:
# ---- TESTE: Req. 05 e 06 ----
# Teste do Req. 05 e Req. 06
exibir_resumo(viagem)
mapa = gerar_mapa(origem, destino, viagem)
print("Mapa salvo em mapa_viagem.html")
mapa

In [ ]:
# ---- TESTE: sem previsão ----
# Teste: falta de previsão (não deve dar erro, só avisar no resumo)
exibir_resumo({**viagem, "previsao": []})

In [ ]:
# ---- TESTE: Req. 07 ----
# Teste do Req. 07: cada caso deve imprimir um aviso claro, sem quebrar o programa
casos = {
    "coordenada com texto":      lambda: validar_coord(("abc", -46.6), "origem"),
    "coordenada incompleta":     lambda: validar_coord((-23.5,), "destino"),
    "coordenada fora do mapa":   lambda: validar_coord((200, -46.6), "origem"),
    "coordenada vazia (None)":   lambda: validar_coord(None, "destino"),
    "resposta vazia":            lambda: exigir([], "Nenhuma linha encontrada para '9999'."),
    "falha na API (sem login)":  lambda: requisitar("/Linha/Buscar", termosBusca="8000"),
}

for nome, teste in casos.items():
    try:
        teste()
        print(f"[{nome}] passou sem erro (inesperado para este teste)")
    except ErroViagem as e:
        print(f"[{nome}] ⚠️ {e}")